# Частина 1. Перевірка ідеї: правила (Baseline) проти LLM та верифікація контракту

Цей notebook виконує дослід **до реалізації HTTP-сервісу**.
Мета: зафіксувати структуру полів задачі, перевірити вузький baseline на регулярних виразах, порівняти дві інструкції моделі (A/B) та протестувати детермінований захист від галюцинацій на 10 синтетичних випадках.

In [1]:
import json
import os
import re
from typing import Any
from pydantic import BaseModel, Field, ValidationError
from IPython.display import Markdown, display


## 1. Продуктовий контракт полів та 10 синтетичних повідомлень

Поля задачі:
- `title` (рядок, 3–120 символів) — назва задачі;
- `description` (рядок або `null`, до 500 символів) — короткий опис;
- `assignee` (рядок) — відповідальний-псевдонім (має бути буквально присутній у тексті);
- `due_date` (рядок `YYYY-MM-DD`) — строк виконання (має бути буквально присутній у тексті).

In [2]:
SYNTHETIC_CASES = [
    {
        "id": "M01",
        "category": "normal_strict",
        "text": "Підготувати демо; відповідальний: Ірина; строк: 2026-10-09",
        "expected_status": "created",
        "expected_assignee": "Ірина",
        "expected_due_date": "2026-10-09",
    },
    {
        "id": "M02",
        "category": "paraphrased",
        "text": "Домовилися, що до 2026-10-15 Максим оновить документацію API для клієнтів",
        "expected_status": "created",
        "expected_assignee": "Максим",
        "expected_due_date": "2026-10-15",
    },
    {
        "id": "M03",
        "category": "normal_with_desc",
        "text": "Налаштувати бекапи БД: щоденний дамп о 03:00; відповідальний: Олег; строк: 2026-10-20",
        "expected_status": "created",
        "expected_assignee": "Олег",
        "expected_due_date": "2026-10-20",
    },
    {
        "id": "M04",
        "category": "missing_due_date",
        "text": "Перевірити логування помилок на тестовому сервері; відповідальний: Ірина",
        "expected_status": "needs_clarification",
        "expected_assignee": None,
        "expected_due_date": None,
    },
    {
        "id": "M05",
        "category": "missing_assignee",
        "text": "Написати інструкцію для нових менторів до 2026-10-18",
        "expected_status": "needs_clarification",
        "expected_assignee": None,
        "expected_due_date": None,
    },
    {
        "id": "M06",
        "category": "contradiction_dates",
        "text": "Зібрати фідбек від користувачів; відповідальний: Анна; строк: 2026-10-12 або 2026-10-19",
        "expected_status": "needs_clarification",
        "expected_assignee": None,
        "expected_due_date": None,
    },
    {
        "id": "M07",
        "category": "contradiction_assignees",
        "text": "Провести код-рев'ю модуля авторизації; відповідальний: Ірина; відповідальний: Максим; строк: 2026-10-11",
        "expected_status": "needs_clarification",
        "expected_assignee": None,
        "expected_due_date": None,
    },
    {
        "id": "M08",
        "category": "hallucinated_proposal",
        "text": "Оновити залежності проєкту; відповідальний: Олег; строк: 2026-10-22",
        "expected_status": "rejected_ungrounded",
        "expected_assignee": None,
        "expected_due_date": None,
    },
    {
        "id": "M09",
        "category": "invalid_structure",
        "text": "Підготувати звіт по спринту; відповідальний: Анна; строк: 2026-10-25",
        "expected_status": "invalid_structure",
        "expected_assignee": None,
        "expected_due_date": None,
    },
    {
        "id": "M10",
        "category": "provider_unavailable",
        "text": "Забронювати аудиторію для ретроспективи; відповідальний: Максим; строк: 2026-10-14",
        "expected_status": "provider_unavailable",
        "expected_assignee": None,
        "expected_due_date": None,
    },
]


## 2. Вузький Baseline на правилах (без LLM)

Правило шукає лише суворий формат `відповідальний: <Ім'я>` та `строк: <YYYY-MM-DD>`. Перевіряємо, на яких повідомленнях простий парсер працює, а де потрібен гнучкий розбір мовною моделлю.

In [3]:
DATE_RE = re.compile(r"\b(\d{4}-\d{2}-\d{2})\b")
ASSIGNEE_RE = re.compile(r"відповідальний:\s*([А-ЯІЇЄҐа-яіїєґA-Za-z0-9_\-]+)", re.IGNORECASE)

def baseline_rule_parser(text: str) -> dict[str, Any]:
    dates = DATE_RE.findall(text)
    assignees = ASSIGNEE_RE.findall(text)
    if len(dates) > 1 or len(assignees) > 1:
        return {"status": "needs_clarification", "reason": "contradiction"}
    if len(dates) == 0 or len(assignees) == 0:
        return {"status": "needs_clarification", "reason": "missing_fields"}
    title_part = text.split(";")[0].strip()
    return {
        "status": "created",
        "title": title_part,
        "assignee": assignees[0],
        "due_date": dates[0],
    }


## 3. A/B порівняння інструкцій моделі та шляхи OpenAI / OpenRouter

Порівнюємо дві системні інструкції, які відрізняються **одним поясненим елементом** — явною забороною вгадувати відсутні поля:
- **Prompt A (базовий):** просить виділити JSON-поля з тексту без вказівки щодо `null` при відсутності даних.
- **Prompt B (уточнений контракт):** додає суворе правило: *«Якщо відповідального або дати YYYY-MM-DD немає у вхідному тексті дослівно, поверни для них null. Заборонено підставляти сьогоднішню дату чи вигадувати виконавця»*.

Конфігурація клієнта підтримує `openai` (`gpt-4o-mini`) та `openrouter` (`nvidia/nemotron-3-super-120b-a12b:free`) через OpenAI Python SDK. Ключі зчитуються лише з оточення (`.env`).

In [4]:
PROMPT_A = """Виділи із повідомлення задачу та поверни JSON з полями: title, description, assignee, due_date (YYYY-MM-DD)."""

PROMPT_B = """Виділи із повідомлення задачу та поверни лише JSON з полями: title, description, assignee, due_date (YYYY-MM-DD).
КРИТИЧНЕ ПРАВИЛО: Якщо відповідального або дати YYYY-MM-DD немає у вхідному тексті дослівно, поверни для них null. Заборонено підставляти сьогоднішню дату чи вигадувати виконавця."""

def build_llm_client(provider: str = "openrouter"):
    from openai import OpenAI
    if provider == "openai":
        api_key = os.getenv("OPENAI_API_KEY", "")
        model = "gpt-4o-mini"
        base_url = None
    elif provider == "openrouter":
        api_key = os.getenv("OPENROUTER_API_KEY", "")
        model = "nvidia/nemotron-3-super-120b-a12b:free"
        base_url = "https://openrouter.ai/api/v1"
    else:
        raise ValueError(f"Невідомий провайдер: {provider}")
    if not api_key:
        return None, model
    return OpenAI(api_key=api_key, base_url=base_url, timeout=20.0, max_retries=0), model


## 4. Детермінована перевірка структури та підтвердження полів текстом (Grounding)

Програма не довіряє виходу моделі наосліп. Навіть якщо JSON синтаксично правильний, код перевіряє:
1. Чи немає у вихідному повідомленні суперечностей (більше однієї дати або більше одного маркера відповідального).
2. Чи присутні обидва обов'язкові поля (`assignee` та `due_date`).
3. Чи буквально містяться запропоновані `assignee` та `due_date` у вхідному тексті.

In [5]:
class LLMProposal(BaseModel):
    title: str | None = None
    description: str | None = None
    assignee: str | None = None
    due_date: str | None = None

def verify_proposal_against_text(text: str, raw_response: str | Exception) -> dict[str, Any]:
    if isinstance(raw_response, Exception):
        return {"status": "provider_unavailable", "reason": str(raw_response)}
    try:
        data = json.loads(raw_response)
        proposal = LLMProposal.model_validate(data)
    except (json.JSONDecodeError, ValidationError) as err:
        return {"status": "invalid_structure", "reason": type(err).__name__}

    # 1. Перевірка суперечностей у самому вхідному тексті
    dates_in_text = DATE_RE.findall(text)
    assignees_in_text = ASSIGNEE_RE.findall(text)
    if len(set(dates_in_text)) > 1 or len(set(assignees_in_text)) > 1:
        return {"status": "needs_clarification", "reason": "contradictory_input"}

    # 2. Перевірка повноти обов'язкових полів
    if not proposal.title or not proposal.assignee or not proposal.due_date:
        return {"status": "needs_clarification", "reason": "missing_required_fields"}

    # 3. Перевірка формату дати та буквального підтвердження текстом (Grounding)
    if not DATE_RE.fullmatch(proposal.due_date):
        return {"status": "rejected_ungrounded", "reason": "invalid_date_format"}
    if proposal.assignee not in text or proposal.due_date not in text:
        return {"status": "rejected_ungrounded", "reason": "hallucinated_field_not_in_text"}

    return {
        "status": "created",
        "title": proposal.title.strip(),
        "description": proposal.description,
        "assignee": proposal.assignee.strip(),
        "due_date": proposal.due_date,
    }


In [6]:
# Явно позначені підготовлені відповіді (synthetic fixtures) для детермінованої перевірки
# Використовуються для перевірки логіки валідатора незалежно від мережі
PREPARED_RESPONSES_PROMPT_B = {
    "M01": json.dumps({"title": "Підготувати демо", "description": None, "assignee": "Ірина", "due_date": "2026-10-09"}, ensure_ascii=False),
    "M02": json.dumps({"title": "Оновити документацію API для клієнтів", "description": None, "assignee": "Максим", "due_date": "2026-10-15"}, ensure_ascii=False),
    "M03": json.dumps({"title": "Налаштувати бекапи БД", "description": "щоденний дамп о 03:00", "assignee": "Олег", "due_date": "2026-10-20"}, ensure_ascii=False),
    "M04": json.dumps({"title": "Перевірити логування помилок", "description": None, "assignee": "Ірина", "due_date": None}, ensure_ascii=False),
    "M05": json.dumps({"title": "Написати інструкцію для нових менторів", "description": None, "assignee": None, "due_date": "2026-10-18"}, ensure_ascii=False),
    "M06": json.dumps({"title": "Зібрати фідбек від користувачів", "description": None, "assignee": "Анна", "due_date": "2026-10-12"}, ensure_ascii=False),
    "M07": json.dumps({"title": "Провести код-рев'ю модуля авторизації", "description": None, "assignee": "Ірина", "due_date": "2026-10-11"}, ensure_ascii=False),
    "M08": json.dumps({"title": "Оновити залежності проєкту", "description": None, "assignee": "Марія", "due_date": "2026-10-22"}, ensure_ascii=False),
    "M09": "НЕВАЛІДНИЙ JSON {{{",
    "M10": RuntimeError("OpenRouter API 503 Service Unavailable"),
}

# Демонстрація різниці A/B на M04 (відсутній строк): Prompt A галюцинує дату, але наш код-валідатор її все одно перехоплює
PROMPT_A_ON_M04 = json.dumps({"title": "Перевірити логування помилок", "description": None, "assignee": "Ірина", "due_date": "2026-10-10"}, ensure_ascii=False)
ab_check_a = verify_proposal_against_text(SYNTHETIC_CASES[3]["text"], PROMPT_A_ON_M04)
ab_check_b = verify_proposal_against_text(SYNTHETIC_CASES[3]["text"], PREPARED_RESPONSES_PROMPT_B["M04"])
print(f"M04 з Prompt A (модель вигадала дату): валідатор повернув -> {ab_check_a['status']} ({ab_check_a['reason']})")
print(f"M04 з Prompt B (модель чесно повернула null): валідатор повернув -> {ab_check_b['status']} ({ab_check_b['reason']})")


M04 з Prompt A (модель вигадала дату): валідатор повернув -> rejected_ungrounded (hallucinated_field_not_in_text)
M04 з Prompt B (модель чесно повернула null): валідатор повернув -> needs_clarification (missing_required_fields)


In [7]:
client, model_name = build_llm_client("openrouter")
live_status = "live_executed" if client is not None else "not_run (ключ OPENROUTER_API_KEY не задано, використано детерміновані фікстури)"
print(f"Статус живого виклику LLM: {live_status} | Цільова модель: openrouter / {model_name} | Бюджет: $0.00 (free tier)")

table_md = "| ID | Категорія | Очікування | Baseline (Правила) | LLM + Валідатор коду | Збіг із контрактом |\n|---|---|---|---|---|---|\n"
for case in SYNTHETIC_CASES:
    cid = case["id"]
    b_res = baseline_rule_parser(case["text"])["status"]
    v_res = verify_proposal_against_text(case["text"], PREPARED_RESPONSES_PROMPT_B[cid])["status"]
    match = "PASS" if v_res == case["expected_status"] else "FAIL"
    assert match == "PASS", f"Помилка контракту на випадку {cid}"
    table_md += f"| {cid} | `{case['category']}` | `{case['expected_status']}` | `{b_res}` | `{v_res}` | **{match}** |\n"

display(Markdown(table_md))


Статус живого виклику LLM: not_run (ключ OPENROUTER_API_KEY не задано, використано детерміновані фікстури) | Цільова модель: openrouter / nvidia/nemotron-3-super-120b-a12b:free | Бюджет: $0.00 (free tier)


| ID | Категорія | Очікування | Baseline (Правила) | LLM + Валідатор коду | Збіг із контрактом |
|---|---|---|---|---|---|
| M01 | `normal_strict` | `created` | `created` | `created` | **PASS** |
| M02 | `paraphrased` | `created` | `needs_clarification` | `created` | **PASS** |
| M03 | `normal_with_desc` | `created` | `created` | `created` | **PASS** |
| M04 | `missing_due_date` | `needs_clarification` | `needs_clarification` | `needs_clarification` | **PASS** |
| M05 | `missing_assignee` | `needs_clarification` | `needs_clarification` | `needs_clarification` | **PASS** |
| M06 | `contradiction_dates` | `needs_clarification` | `needs_clarification` | `needs_clarification` | **PASS** |
| M07 | `contradiction_assignees` | `needs_clarification` | `needs_clarification` | `needs_clarification` | **PASS** |
| M08 | `hallucinated_proposal` | `rejected_ungrounded` | `created` | `rejected_ungrounded` | **PASS** |
| M09 | `invalid_structure` | `invalid_structure` | `created` | `invalid_structure` | **PASS** |
| M10 | `provider_unavailable` | `provider_unavailable` | `created` | `provider_unavailable` | **PASS** |


## 5. Висновок досліду для специфікації сервісу

1. **Чому правил (Baseline) недостатньо:** регулярні вирази ідеально працюють на жорсткому форматі (`M01`, `M03`), але падають на природному перефразуванні (`M02`: *«Домовилися, що до 2026-10-15 Максим оновить...»*), повертаючи хибне `needs_clarification`.
2. **Чому самої лише LLM недостатньо:** без детермінованої перевірки в коді модель може вигадати відсутню дату (як у `Prompt A` на `M04`), підставити чуже ім'я (`M08`) або обрати випадкову дату з двох суперечливих (`M06`).
3. **Рішення для специфікації (Частина 2):** використовуємо гібридний конвеєр `Prompt B -> JSON Schema Validation -> Text Grounding Policy`. Якщо модель повертає `null` або в тексті є суперечність — сервіс повертає HTTP `422` зі статусом `needs_clarification`. Якщо модель вигадує поле, якого немає в тексті — сервіс відхиляє запит зі статусом `rejected_ungrounded` без зміни сховища.